# RSNA Knee — 03: Build compact training volumes from the DICOMs (CPU only)

**Settings:** Accelerator **None (CPU)** · Internet **On** (installs JPEG decoders) · Input: the competition data.
Expect roughly 1.5–4 hours. Use **Save Version → Save & Run All**.

**What it does:** for every training study, picks the best series for each of 5 slots
(sagittal fluid / sagittal non-fluid / coronal fluid / coronal non-fluid / axial fluid), puts every series into the same
standard orientation, crops a constant 140 mm field of view, resizes to 256×256, keeps 20 evenly spaced slices,
and stores the result as uint8. 570 GB of DICOMs → about 10–12 GB the training notebook can read quickly.

**Outputs** in `/kaggle/working/`: `vols/shard_XXX.npz` (one array per study), `vol_index.csv`, `prep_summary.json`,
and `preview.png` (a few slices, for you to eyeball — don't share it, it is competition image data).

If the run would exceed Kaggle's 12-hour limit, set `N_PARTS = 2` and run it twice (PART = 0, then a copy with PART = 1).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pyjpegls"],
               check=False)

In [ ]:
# ---- shared preprocessing: identical code is used for training volumes and at submission time ----
import os, glob
import numpy as np
import pydicom
import cv2
from concurrent.futures import ThreadPoolExecutor

# 5 slots = plane x (fluid-sensitive / not). Axial non-fluid exists in only ~19% of studies -> used only as axial fallback.
SLOT_NAMES = ["sag_fs", "sag_nfs", "cor_fs", "cor_nfs", "ax_fs"]
SLOT_INDEX = {("sagittal", 1): 0, ("sagittal", 0): 1, ("coronal", 1): 2, ("coronal", 0): 3, ("axial", 1): 4}
N_SLICES = 20          # slices kept per slot (evenly spaced through the series)
IMG = 256              # output pixels per side
CROP_MM = 140.0        # constant physical field of view -> 0.55 mm/px at 256
MAX_SERIES_SLICES = 80 # prefer 2D series over very long 3D series when both exist

# Standard radiological display per plane, in DICOM patient coords (x = patient left, y = posterior, z = head):
# (axis that image columns run along, its sign, axis that image rows run along, its sign, through-plane slice axis)
TARGET = {"sagittal": (1, +1, 2, -1, 0),   # front of knee on image left, head up
          "coronal":  (0, +1, 2, -1, 1),   # patient's right on image left, head up
          "axial":    (0, +1, 1, +1, 2)}   # patient's right on image left, front up

def plane_of(iop):
    v = np.asarray(iop, dtype=float)
    n = np.cross(v[:3], v[3:6])
    return ["sagittal", "coronal", "axial"][int(np.argmax(np.abs(n)))]

def _read_header(f):
    try:
        ds = pydicom.dcmread(f, stop_before_pixels=True,
                             specific_tags=["ImageOrientationPatient", "ImagePositionPatient", "PixelSpacing",
                                            "Rows", "Columns", "InstanceNumber"])
        return f, [float(x) for x in ds.ImageOrientationPatient], [float(x) for x in ds.ImagePositionPatient], \
               [float(x) for x in ds.PixelSpacing]
    except Exception:
        return f, None, None, None

def _read_pixels(f):
    try:
        ds = pydicom.dcmread(f)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:  # multi-frame or colour: take first frame / channel
            a = a[0] if a.shape[0] < a.shape[-1] else a[..., 0]
        return a, str(ds.file_meta.TransferSyntaxUID) if hasattr(ds, "file_meta") else "?"
    except Exception as ex:
        return None, type(ex).__name__

def series_headers(series_dir, pool):
    files = sorted(glob.glob(os.path.join(series_dir, "*.dcm"))) or \
            sorted(f for f in glob.glob(os.path.join(series_dir, "*")) if os.path.isfile(f))
    hdrs = [h for h in pool.map(_read_header, files) if h[1] is not None]
    if not hdrs:
        return None
    # keep the majority orientation (drops localisers mixed into a series)
    keys = [tuple(np.round(h[1], 1)) for h in hdrs]
    best = max(set(keys), key=keys.count)
    hdrs = [h for h, k in zip(hdrs, keys) if k == best]
    plane = plane_of(hdrs[0][1])
    hdrs.sort(key=lambda h: h[2][TARGET[plane][4]])
    return {"plane": plane, "hdrs": hdrs, "n": len(hdrs)}

def reorient(img, r, c, spacing, plane):
    """Flip/transpose so every series is displayed in the same standard orientation."""
    ra, rs, ca, cs, _ = TARGET[plane]
    r, c = np.asarray(r), np.asarray(c)
    sp = list(spacing)  # [row spacing (between rows), column spacing]
    if int(np.argmax(np.abs(r))) != ra:
        img, r, c, sp = img.T, c, r, sp[::-1]
    if np.sign(r[ra]) != rs:
        img = img[:, ::-1]
    if np.sign(c[ca]) != cs:
        img = img[::-1, :]
    return np.ascontiguousarray(img), sp

def crop_resize(img, sp):
    h, w = img.shape
    hr, hc = CROP_MM / 2 / max(sp[0], 1e-3), CROP_MM / 2 / max(sp[1], 1e-3)
    cy, cx = h / 2, w / 2
    y0, y1, x0, x1 = int(round(cy - hr)), int(round(cy + hr)), int(round(cx - hc)), int(round(cx + hc))
    pad = max(0, -y0, -x0, y1 - h, x1 - w)
    if pad:
        img = np.pad(img, pad)
        y0, y1, x0, x1 = y0 + pad, y1 + pad, x0 + pad, x1 + pad
    crop = img[y0:y1, x0:x1]
    interp = cv2.INTER_AREA if crop.shape[0] > IMG else cv2.INTER_LINEAR
    return cv2.resize(crop, (IMG, IMG), interpolation=interp)

def build_slot(info, pool):
    hdrs = info["hdrs"]
    idx = np.round(np.linspace(0, len(hdrs) - 1, N_SLICES)).astype(int)
    uniq = sorted(set(idx.tolist()))
    pix = dict(zip(uniq, pool.map(lambda i: _read_pixels(hdrs[i][0]), uniq)))
    out, ts = [], set()
    for i in idx:
        a, t = pix[i]
        ts.add(t)
        if a is None:
            out.append(np.zeros((IMG, IMG), np.float32)); continue
        _, iop, _, sp = hdrs[i]
        a, sp2 = reorient(a, iop[:3], iop[3:], sp, info["plane"])
        out.append(crop_resize(a, sp2))
    vol = np.stack(out)
    nz = vol[vol > 0]
    lo, hi = (np.percentile(nz, [1, 99.5]) if nz.size > 100 else (0.0, 1.0))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255
    return vol.astype(np.uint8), ts, sum(pix[i][0] is None for i in uniq)

def process_study(study_dir, series_rows):
    """series_rows: list of (SeriesInstanceUID, Fluid_Sensitive). Returns (vol [5,N,IMG,IMG] uint8, mask [5], info)."""
    vol = np.zeros((len(SLOT_NAMES), N_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros(len(SLOT_NAMES), np.uint8)
    info = {"n_series": len(series_rows), "plane_mismatch": 0, "bad_pixels": 0, "ts": set(), "slot_n": [0] * 5}
    cands = {}
    with ThreadPoolExecutor(8) as pool:
        for uid, fluid in series_rows:
            sdir = os.path.join(study_dir, str(uid))
            if not os.path.isdir(sdir):
                hits = [d for d, _, _ in os.walk(study_dir) if os.path.basename(d) == str(uid)]
                if not hits:
                    continue
                sdir = hits[0]
            h = series_headers(sdir, pool)
            if h is None:
                continue
            key = (h["plane"], int(fluid))
            slot = SLOT_INDEX.get(key, 4 if h["plane"] == "axial" else None)
            if slot is None:
                continue
            fallback = key not in SLOT_INDEX
            score = (not fallback, h["n"] <= MAX_SERIES_SLICES, -abs(h["n"] - 30))
            if slot not in cands or score > cands[slot][0]:
                cands[slot] = (score, h)
        for slot, (_, h) in cands.items():
            v, ts, bad = build_slot(h, pool)
            vol[slot], mask[slot] = v, 1
            info["ts"] |= ts
            info["bad_pixels"] += bad
            info["slot_n"][slot] = h["n"]
    return vol, mask, info

In [ ]:
import json, time, collections
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

N_PARTS, PART = 1, 0
SHARD_SIZE = 100
OUT = os.environ.get("KNEE_OUT_DIR", "/kaggle/working")
INPUT_ROOT = os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input")
os.makedirs(f"{OUT}/vols", exist_ok=True)

def find_file(name):
    for cur, dirs, files in os.walk(INPUT_ROOT, followlinks=True):
        if len(dirs) > 200:
            dirs[:] = []
        if name in files:
            return os.path.join(cur, name)

series_csv = find_file("train_series.csv")
assert series_csv, "Competition data not attached"
DATA = os.path.dirname(series_csv)
IMG_ROOT = os.path.join(DATA, "train_series")
ser = pd.read_csv(series_csv)
groups = {s: list(zip(g.SeriesInstanceUID, g.Fluid_Sensitive)) for s, g in ser.groupby("StudyInstanceUID")}
studies = sorted(groups)
studies = [s for i, s in enumerate(studies) if i % N_PARTS == PART]
shards = [studies[i:i + SHARD_SIZE] for i in range(0, len(studies), SHARD_SIZE)]
print(f"{len(studies)} studies in this part -> {len(shards)} shards. CPUs: {os.cpu_count()}")

def work(uid):
    t = time.time()
    try:
        vol, mask, info = process_study(os.path.join(IMG_ROOT, uid), groups[uid])
        info["error"] = None
    except Exception as ex:
        vol = np.zeros((len(SLOT_NAMES), N_SLICES, IMG, IMG), np.uint8)
        mask = np.zeros(len(SLOT_NAMES), np.uint8)
        info = {"error": f"{type(ex).__name__}: {str(ex)[:80]}", "ts": set(), "bad_pixels": 0, "slot_n": [0] * 5, "n_series": 0}
    info["sec"] = time.time() - t
    return uid, vol, mask, info

In [ ]:
INDEX = f"{OUT}/vol_index.csv"
index = pd.read_csv(INDEX).to_dict("records") if os.path.exists(INDEX) else []
ts_counter, t0, done = collections.Counter(), time.time(), 0
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for k, shard in enumerate(shards):
        sid = PART * 1000 + k
        path = f"{OUT}/vols/shard_{sid:04d}.npz"
        if os.path.exists(path):
            done += len(shard); continue
        arrays, rows = {}, []
        for uid, vol, mask, info in ex.map(work, shard):
            arrays[uid] = vol
            ts_counter.update(info["ts"])
            rows.append({"StudyInstanceUID": uid, "shard": sid, **{f"m_{n}": int(mask[i]) for i, n in enumerate(SLOT_NAMES)},
                         **{f"n_{n}": info["slot_n"][i] for i, n in enumerate(SLOT_NAMES)},
                         "bad_pixels": info["bad_pixels"], "error": info["error"], "sec": round(info["sec"], 2)})
        np.savez_compressed(path, **arrays)
        index += rows
        pd.DataFrame(index).to_csv(INDEX, index=False)
        done += len(shard)
        el = time.time() - t0
        print(f"shard {k+1}/{len(shards)}  studies {done}/{len(studies)}  {el/60:.1f} min  "
              f"~{el/max(done,1)*(len(studies)-done)/60:.0f} min left  shard {os.path.getsize(path)/1e6:.0f} MB")

## Preview (look at it yourself; don't share it)
Check: **sagittal** — kneecap on the left, thigh at the top. **Coronal** — thigh at the top, shin at the bottom.
**Axial** — kneecap at the top. Every row should look the same way round.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
first = np.load(sorted(glob.glob(f"{OUT}/vols/shard_*.npz"))[0])
keys = list(first.keys())[:4]
fig, axes = plt.subplots(len(keys), 5, figsize=(15, 3 * len(keys)))
for r, kk in enumerate(keys):
    v = first[kk]
    for c in range(5):
        axes[r, c].imshow(v[c, N_SLICES // 2], cmap="gray"); axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(SLOT_NAMES[c])
plt.tight_layout(); plt.savefig(f"{OUT}/preview.png", dpi=70); plt.show()

## SUMMARY — paste this back to Claude

In [ ]:
idx = pd.DataFrame(index)
summary = {
    "studies_done": int(len(idx)), "parts": [PART, N_PARTS],
    "errors": int(idx.error.notna().sum()), "error_examples": idx.error.dropna().value_counts().head(5).to_dict(),
    "slot_coverage": {n: round(float(idx[f"m_{n}"].mean()), 3) for n in SLOT_NAMES},
    "studies_with_no_slots": int((idx[[f"m_{n}" for n in SLOT_NAMES]].sum(axis=1) == 0).sum()),
    "slices_per_chosen_series_median": {n: float(idx.loc[idx[f"m_{n}"] == 1, f"n_{n}"].median()) for n in SLOT_NAMES},
    "studies_with_unreadable_slices": int((idx.bad_pixels > 0).sum()),
    "transfer_syntaxes": dict(ts_counter.most_common(8)),
    "sec_per_study_median": float(idx.sec.median()),
    "total_minutes": round((time.time() - t0) / 60, 1),
    "output_gb": round(sum(os.path.getsize(p) for p in glob.glob(f"{OUT}/vols/*.npz")) / 1e9, 2),
}
json.dump(summary, open(f"{OUT}/prep_summary.json", "w"), indent=1, default=str)
print(json.dumps(summary, indent=1, default=str))